# 01 – Scene availability (Task 1)
Which Landsat 8, Sentinel-2 and Sentinel-1 scenes cover the Red River valley during the 2019 flood
(mid-April to early May) and in snow-free reference periods? Used to pick the flood/reference dates
and the AOI, and to document data availability (clouds, temporal gaps) for the report.

In [1]:
import ee
import common as c

c.init()
c.TABLES.mkdir(parents=True, exist_ok=True)

## AOI candidates

In [2]:
m = c.Map()  # geemap map with a basemap that loads in VS Code
for name in c.AOI_BOXES:
    m.addLayer(c.aoi(name), {"color": "red"}, name)
m

Map(center=[47.9, -97.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

## Scene queries
Scene-level cloud cover only for now; AOI-level cloud fraction comes later for the shortlisted dates.

In [3]:
def landsat8(roi, window):
    col = ee.ImageCollection(c.LANDSAT8).filterBounds(roi).filterDate(*window)
    return c.scene_table(col, {"path": "WRS_PATH", "row": "WRS_ROW", "cloud_%": "CLOUD_COVER"})

def sentinel2(roi, window):
    col = ee.ImageCollection(c.SENTINEL2).filterBounds(roi).filterDate(*window)
    return c.scene_table(col, {"tile": "MGRS_TILE", "cloud_%": "CLOUDY_PIXEL_PERCENTAGE"})

def sentinel1(roi, window):
    col = (ee.ImageCollection(c.SENTINEL1).filterBounds(roi).filterDate(*window)
           .filter(ee.Filter.eq("instrumentMode", "IW")))
    return c.scene_table(col, {"pass": "orbitProperties_pass",
                               "rel_orbit": "relativeOrbitNumber_start",
                               "pol": "transmitterReceiverPolarisation"})

## During the flood

In [4]:
tables = {}
for aoi_name in c.AOI_BOXES:
    roi = c.aoi(aoi_name)
    for sensor, fn in [("L8", landsat8), ("S2", sentinel2), ("S1", sentinel1)]:
        df = fn(roi, c.FLOOD_WINDOW)
        tables[(aoi_name, sensor)] = df
        df.to_csv(c.TABLES / f"scenes_flood_{aoi_name}_{sensor}.csv", index=False)
        print(f"{aoi_name:20s} {sensor}: {len(df)} scenes")

fargo_grandforks     L8: 9 scenes
fargo_grandforks     S2: 30 scenes
fargo_grandforks     S1: 8 scenes
grandforks_pembina   L8: 6 scenes
grandforks_pembina   S2: 20 scenes
grandforks_pembina   S1: 8 scenes


In [5]:
tables[("grandforks_pembina", "L8")]

,date,path,row,cloud_%
0,2019-04-15 17:15,30,26,95.38
1,2019-04-15 17:16,30,27,97.96
2,2019-04-22 17:21,31,26,1.35
3,2019-04-22 17:22,31,27,0.89
4,2019-05-08 17:22,31,26,91.07
5,2019-05-08 17:22,31,27,100.00


In [6]:
tables[("grandforks_pembina", "S2")].query("`cloud_%` < 30")

,date,tile,cloud_%
0,2019-04-13 17:30,14UPU,18.643191
1,2019-04-13 17:30,14UPV,0.364786
2,2019-04-16 17:40,14UPU,3.459325
3,2019-04-16 17:40,14UPV,3.104548
8,2019-04-26 17:40,14UPU,17.781623


In [7]:
tables[("grandforks_pembina", "S1")]

,date,pass,rel_orbit,pol
0,2019-04-17 00:14,ASCENDING,63,"[VV, VH]"
1,2019-04-17 00:14,ASCENDING,63,"[VV, VH]"
2,2019-04-22 00:22,ASCENDING,136,"[VV, VH]"
3,2019-04-22 00:23,ASCENDING,136,"[VV, VH]"
4,2019-04-29 00:14,ASCENDING,63,"[VV, VH]"
5,2019-04-29 00:14,ASCENDING,63,"[VV, VH]"
6,2019-05-04 00:22,ASCENDING,136,"[VV, VH]"
7,2019-05-04 00:23,ASCENDING,136,"[VV, VH]"


## Snow-free reference periods

In [8]:
for aoi_name in c.AOI_BOXES:
    roi = c.aoi(aoi_name)
    for label, window in [("autumn2018", c.REF_WINDOW_AUTUMN), ("summer2019", c.REF_WINDOW_SUMMER)]:
        for sensor, fn in [("L8", landsat8), ("S2", sentinel2), ("S1", sentinel1)]:
            df = fn(roi, window)
            df.to_csv(c.TABLES / f"scenes_ref_{label}_{aoi_name}_{sensor}.csv", index=False)
            clear = (df["cloud_%"] < 20).sum() if "cloud_%" in df else "-"
            print(f"{aoi_name:20s} {label:11s} {sensor}: {len(df):3d} scenes, clear(<20%): {clear}")

fargo_grandforks     autumn2018  L8:  21 scenes, clear(<20%): 10
fargo_grandforks     autumn2018  S2:  69 scenes, clear(<20%): 22
fargo_grandforks     autumn2018  S1:  20 scenes, clear(<20%): -
fargo_grandforks     summer2019  L8:  40 scenes, clear(<20%): 22
fargo_grandforks     summer2019  S2: 108 scenes, clear(<20%): 39
fargo_grandforks     summer2019  S1:  30 scenes, clear(<20%): -
grandforks_pembina   autumn2018  L8:  13 scenes, clear(<20%): 9
grandforks_pembina   autumn2018  S2:  46 scenes, clear(<20%): 12
grandforks_pembina   autumn2018  S1:  20 scenes, clear(<20%): -
grandforks_pembina   summer2019  L8:  22 scenes, clear(<20%): 16
grandforks_pembina   summer2019  S2:  73 scenes, clear(<20%): 28
grandforks_pembina   summer2019  S1:  30 scenes, clear(<20%): -


## Notes / decision
*(fill in after running: chosen flood date(s), reference date(s), AOI, S1 orbit pair)*